# Decision Tree Regression - California Housing Dataset

## Step 1: Import Required Libraries

We need to import essential libraries for data manipulation, visualization, and machine learning:
- **numpy**: For numerical operations
- **pandas**: For data manipulation and analysis
- **matplotlib**: For plotting and visualization
- **sklearn**: For machine learning algorithms and datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

## Step 2: Load the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["Price"] = housing.target

df.head()

## Step 3: Data Exploration

Understanding the dataset structure and statistics is crucial before building any model.

In [ ]:
df.shape
df.info()
df.describe()

## Step 4: Check for Missing Values

In [ ]:
df.isnull().sum()

## Step 5: Check for Duplicate Rows

In [ ]:
df.duplicated().sum()

## Step 6: Split Features and Target

In [ ]:
X = df.drop("Price",axis=1)

y = df["Price"]

## Step 7: Data Visualization - Histograms

In [ ]:
df.hist(
    figsize=(14,10),
    bins=30
)

plt.show()

## Step 8: Correlation Analysis

In [ ]:
corr=df.corr()

corr["Price"].sort_values(
    ascending=False
)
import seaborn as sns

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.show()

## Step 9: Outlier Detection - Boxplots

In [ ]:
for col in df.columns:

    plt.figure()

    plt.boxplot(df[col])

    plt.title(col)

    plt.show()

## Step 10: Feature Scaling (Optional for Decision Trees)

**Note:** Decision trees are scale-invariant, meaning they don't require feature scaling. However, we'll still scale for consistency with other models and for visualization purposes.

**StandardScaler Formula:**
$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

X_scaled=pd.DataFrame(
    X_scaled,
    columns=X.columns
)

## Step 11: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=\
train_test_split(
X_scaled,
y,
test_size=0.2,
random_state=42
)
X_train.shape
X_test.shape

## Decision Tree Regression Formula and Concepts

**Decision Tree Structure:**

A decision tree for regression works by:
1. **Splitting**: Dividing the data based on feature values to minimize variance
2. **Prediction**: Taking the mean of target values in each leaf node

**Splitting Criterion (MSE):**

At each node, the tree selects the best split by minimizing:

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \bar{y})^2$$

Where:
- **yᵢ** = actual target value
- **ȳ** = mean of target values in the node
- **n** = number of samples in the node

**Key Concepts:**
- **Root Node**: The top node containing all data
- **Internal Nodes**: Nodes that split the data based on feature thresholds
- **Leaf Nodes**: Terminal nodes that make predictions (mean of samples)
- **Max Depth**: Maximum depth of the tree (controls complexity)
- **Min Samples Split**: Minimum samples required to split a node
- **Min Samples Leaf**: Minimum samples required in a leaf node

**Advantages:**
- No need for feature scaling
- Can capture non-linear relationships
- Easy to interpret and visualize
- Handles both numerical and categorical data

**Disadvantages:**
- Prone to overfitting (needs pruning)
- Unstable (small data changes can create different trees)
- Can create biased trees if some classes dominate

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Train Decision Tree with Default Parameters

In [ ]:
dt = DecisionTreeRegressor(random_state=42)

dt.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = dt.predict(
    X_test
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse**0.5

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R²:",r2)

## Tree Information

In [ ]:
print("Tree Depth:", dt.get_depth())
print("Number of Leaves:", dt.get_n_leaves())
print("Number of Nodes:", dt.tree_.node_count)

## Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": dt.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Decision Tree Feature Importance")
plt.tight_layout()
plt.show()

## Visualization: Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()]
)

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.title(
    "Decision Tree Regression"
)

plt.show()

## Residual Plot

In [ ]:
residuals = y_test-y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals
)

plt.axhline(y=0)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Residuals"
)

plt.title(
    "Residual Plot"
)

plt.show()

## Hyperparameter Tuning

Let's tune the decision tree to prevent overfitting by controlling:
- **max_depth**: Maximum depth of the tree
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples required in a leaf node

In [ ]:
max_depths = [3, 5, 7, 10, 15, 20, None]

results = []

for depth in max_depths:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    
    results.append({
        'Max Depth': depth if depth else 'None',
        'R²': r2,
        'Tree Depth': model.get_depth(),
        'Number of Leaves': model.get_n_leaves()
    })

results_df = pd.DataFrame(results)
print(results_df)

## Visualize Performance vs Max Depth

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(len(results_df)), results_df['R²'], 'o-', label='R² Score')
plt.xticks(range(len(results_df)), results_df['Max Depth'])
plt.xlabel('Max Depth')
plt.ylabel('R² Score')
plt.title('Decision Tree Performance vs Max Depth')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Train Optimized Decision Tree

Based on the results, let's train a tree with the optimal max_depth.

In [ ]:
# Using max_depth=7 as it seems to be a good balance
dt_optimized = DecisionTreeRegressor(max_depth=7, random_state=42)
dt_optimized.fit(X_train, y_train)
y_pred_opt = dt_optimized.predict(X_test)

r2_opt = r2_score(y_test, y_pred_opt)
print(f"Optimized Decision Tree R²: {r2_opt:.4f}")
print(f"Tree Depth: {dt_optimized.get_depth()}")
print(f"Number of Leaves: {dt_optimized.get_n_leaves()}")

## Summary

Decision Tree Regression provides:
- **Non-linear modeling**: Can capture complex non-linear relationships
- **No scaling required**: Works with raw features
- **Interpretability**: Easy to understand and visualize
- **Feature importance**: Identifies most important features

**Key considerations:**
- Prone to overfitting (needs depth control)
- Unstable (sensitive to small data changes)
- May not perform as well as ensemble methods

**Best practices:**
- Use cross-validation to tune hyperparameters
- Limit tree depth to prevent overfitting
- Consider ensemble methods (Random Forest, Gradient Boosting) for better performance